# 03.03 — CypherQuery: Simple Path, Catalogue Registration, and Executor

This notebook demonstrates the **simple / YAML path** (`CypherQuery`) end-to-end:

1. Load queries from YAML and register them in `QueryCatalogue`.
2. Run `validate_query_catalogue` — including catching a renamed-label error.
3. Execute a `CypherQuery` via `CypherExecutor` using a **test double** (no DB).
4. Execute against a **real Neo4j database** (optional — requires `--neo4j`).

## Two authoring paths — quick reference

| | Simple path (`CypherQuery`) | Typed path (`TypedCypherReadQueryModel` / `TypedCypherWriteQueryModel`) |
|---|---|---|
| **Author as** | Direct instance or YAML | Subclass |
| **Params** | Optional `Params` model or name-lists | Required `Params` model |
| **Output** | None — raw `list[dict]` rows | Required `Output` model |
| **Validation** | `validate_cypher_spec` (shared core) | `validate_cypher_spec` (same shared core) |
| **Catalogue** | `register_cypher_query(q)` | `register_read(q)` / `register_write(q)` |
| **Executor** | `CypherExecutor` directly | `CypherExecutor` directly |
| **When to use** | YAML catalogues, on-ramp, script tooling | Application layer, typed contracts |

In [1]:
from shared.filmography import ActedIn, Movie, Person

from orthograph.definition import GraphDefinition
from orthograph.execution import (
    CypherExecutor,
    CypherWriteResultSummary,
)
from orthograph.queries import (
    CypherQuery,
    NoIdentifiers,
    check_syntax,
    load_catalogue,
    validate_catalogue,
)


print("Imports OK")

Imports OK


## 1. Build the domain model

We use the filmography domain throughout: `Person`, `Movie`, and `ActedIn`.

In [2]:
definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)

print(f"Nodes   : {[nt.__label__ for nt in definition.node_types]}")
print(f"Rels    : {[rt.__label__ for rt in definition.relationship_types]}")

Nodes   : ['Person', 'Movie']
Rels    : ['ACTED_IN']


## 2. Load queries from YAML and register in `QueryCatalogue`

`load_query_catalogue` parses a YAML string (or file path) into a list of `CypherQuery` instances.
Register each one with `catalogue.register_cypher_query(q)` to make it a catalogue citizen.

In [3]:
YAML_CATALOGUE = """
- query_id: find_movie_by_title
  cypher_template: "MATCH (m:Movie {title: $title}) RETURN m.title, m.released"
  params_schema:
    type: object
    properties:
      title: {type: string}
    required: [title]
  description: Find a movie by exact title match

- query_id: actor_filmography
  cypher_template: "MATCH (p:Person {name: $name})-[:ACTED_IN]->(m:Movie) RETURN p.name, m.title"
  params_schema:
    type: object
    properties:
      name: {type: string}
    required: [name]
  description: All movies an actor appeared in
"""

catalogue = load_catalogue(YAML_CATALOGUE)

print(f"Registered {len(catalogue.names())} queries: {catalogue.names()}")

Registered 2 queries: ['find_movie_by_title', 'actor_filmography']


## 3. `validate_query_catalogue` — green run

`validate_query_catalogue` iterates every registered query — including `CypherQuery` instances —
and calls the shared `validate_cypher_spec` core.  Both queries above reference valid labels
and relationship types, so the result should be clean.

In [4]:
result = validate_catalogue(catalogue, definition)

print(f"is_valid : {result.is_valid}")
print(f"issues   : {len(result.issues)}")
for issue in result.issues:
    print(f"  [{issue.severity.value}] {issue.code}: {issue.message}")

assert result.is_valid, "Expected clean validation"

is_valid : True
issues   : 0


## 4. Caught renamed-label error (the MP Phase-1 scenario)

A YAML query that references a label not present in the `GraphDefinition` now produces
a `QUERY_UNKNOWN_NODE_LABEL` ERROR — the same code the typed path would emit.

This is the core of the MP Phase-1 goal: CI-time validation of YAML queries.

In [5]:
# Simulate a query where the label was renamed (Film instead of Movie).
stale_yaml = """
- query_id: find_film_stale
  cypher_template: "MATCH (f:Film {title: $title}) RETURN f.title"
  params_schema:
    type: object
    properties:
      title: {type: string}
    required: [title]
  description: Uses old label Film (renamed to Movie)
"""

stale_catalogue = load_catalogue(stale_yaml)

stale_result = validate_catalogue(stale_catalogue, definition)

print(f"is_valid : {stale_result.is_valid}")
for issue in stale_result.issues:
    print(f"  [{issue.severity.value}] {issue.code}: {issue.message}")

assert not stale_result.is_valid
assert any(i.code == "QUERY_UNKNOWN_NODE_LABEL" for i in stale_result.issues)

is_valid : False
  [error] QUERY_UNKNOWN_NODE_LABEL: Query references node label 'Film' not in model


## 5. Stale `$param` caught statically

A `$param` in the Cypher that is not listed in `query_args_required` /
`query_args_optional` is caught at `validate_query_catalogue` (or
`CypherQuery.validate_query(None)`) time — not at runtime by the driver.

In [6]:
# $released is used in the Cypher but not declared in params_schema.
from pydantic import BaseModel


class StaleParam(BaseModel):
    title: str


stale_param_query = CypherQuery(
    query_id="stale_param",
    cypher_template="MATCH (m:Movie {released: $released}) RETURN m.title",
    params_schema=StaleParam,  # wrong — should include 'released' field
    identifiers_schema=NoIdentifiers,
)

# Syntactic-only check (no definition)
syntax_result = check_syntax(stale_param_query)
print(f"is_valid (syntax-only): {syntax_result.is_valid}")
for issue in syntax_result.issues:
    print(f"  [{issue.severity.value}] {issue.code}: {issue.message}")

assert not syntax_result.is_valid
assert any(i.code == "QUERY_PARAM_ALIGNMENT_ERROR" for i in syntax_result.issues)

is_valid (syntax-only): False
  [error] QUERY_PARAM_ALIGNMENT_ERROR: Query 'stale_param': cypher_template uses parameter(s) ['$released'] not declared on stale_param
  [error] QUERY_PARAM_ALIGNMENT_ERROR: Query 'stale_param': stale_param declares field(s) ['$title'] with no matching placeholder in cypher_template


## 6. `CypherExecutor` round-trip (fake session — no DB)

The simple path executes via the **existing** `CypherExecutor` — no duplicated session logic.
Identifiers are bound at construction; the query is passed directly to the executor.
Results are `list[dict]` (raw rows).

In [7]:
from typing import Any


# Minimal fake session (mirrors tests/cypher/test_query_execution.py)
class FakeSession:
    def __init__(self, records: list[dict[str, Any]]) -> None:
        self._records = records

    def __enter__(self) -> "FakeSession":
        return self

    def __exit__(self, *exc: object) -> None:
        return None

    def run(self, cypher: str, **params: Any) -> list[dict[str, Any]]:
        return list(self._records)


FAKE_RECORDS = [
    {"m.title": "The Matrix", "m.released": 1999},
    {"m.title": "Speed", "m.released": 1994},
]


class FindMovieParams(BaseModel):
    title: str


read_query = CypherQuery(
    query_id="find_movie_by_title",
    cypher_template="MATCH (m:Movie {title: $title}) RETURN m.title, m.released",
    params_schema=FindMovieParams,
    identifiers_schema=NoIdentifiers,
)

# adapter removed (E60.3); using query directly
executor = CypherExecutor(lambda: FakeSession(FAKE_RECORDS))

rows = executor.read(read_query, FindMovieParams(title="The Matrix"))

print(f"Rows returned: {len(rows)}")
for row in rows:
    print(f"  {row}")

assert rows == FAKE_RECORDS
assert all(isinstance(r, dict) for r in rows)

Rows returned: 2
  {'m.title': 'The Matrix', 'm.released': 1999}
  {'m.title': 'Speed', 'm.released': 1994}


## 7. Write round-trip (fake session)

Write queries return the full `CypherWriteResultSummary` (raw counters).
The caller reads whichever counter(s) they need.

In [8]:
class FakeWriteCounters:
    def __init__(self, nodes_created: int) -> None:
        self.nodes_created = nodes_created
        self.nodes_deleted = 0
        self.relationships_created = 0
        self.relationships_deleted = 0
        self.properties_set = 0


class FakeWriteResult:
    def __init__(self, nodes_created: int) -> None:
        self._counters = FakeWriteCounters(nodes_created)

    def consume(self) -> "FakeWriteResult":
        return self

    @property
    def counters(self) -> FakeWriteCounters:
        return self._counters


class FakeWriteSession:
    def __enter__(self) -> "FakeWriteSession":
        return self

    def __exit__(self, *exc: object) -> None:
        return None

    def run(self, cypher: str, **params: Any) -> FakeWriteResult:
        return FakeWriteResult(nodes_created=1)


class CreateMovieParams(BaseModel):
    title: str
    released: int


write_query = CypherQuery(
    query_id="create_movie",
    cypher_template="CREATE (m:Movie {title: $title, released: $released})",
    params_schema=CreateMovieParams,
    identifiers_schema=NoIdentifiers,
)

write_executor = CypherExecutor(lambda: FakeWriteSession())

summary = write_executor.write(
    write_query, CreateMovieParams(title="Inception", released=2010)
)

print(f"nodes_created      : {summary.nodes_created}")
print(f"properties_set     : {summary.properties_set}")
print(f"Summary type       : {type(summary).__name__}")

assert isinstance(summary, CypherWriteResultSummary)
assert summary.nodes_created == 1

nodes_created      : 1
properties_set     : 0
Summary type       : CypherWriteResultSummary


---

## 8. Live database (optional)

> **Skip this section** if you do not have a running Neo4j instance.
>
> To run via `pytest --nbval-lax`, add `--neo4j` to the command.  The notebook
> is gated in `notebooks/conftest.py` and is excluded from the default CI run.
>
> Credentials are resolved from (in order):
> 1. `NEO4J_URI` / `NEO4J_USER` / `NEO4J_PASSWORD` environment variables
> 2. `.env` file at the repo root (git-ignored)
> 3. `.env_default` file at the repo root (committed defaults)
> 4. Hard-coded fallbacks below

The key point this section demonstrates: **switching from a fake session to a real driver
session requires only changing the factory passed to `CypherExecutor`.  The adapter,
the query, and the catalogue code are unchanged.**

In [9]:
from neo4j import GraphDatabase
from shared.utils import load_env


neo4j_uri = load_env("NEO4J_URI", "bolt://localhost:7687")
neo4j_user = load_env("NEO4J_USER", "neo4j")
neo4j_password = load_env("NEO4J_PASSWORD", "password")

print(f"Connecting to {neo4j_uri} as {neo4j_user} ...")
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
driver.verify_connectivity()
print("Connected")

Connecting to bolt://localhost:7687 as neo4j ...
Connected


In [10]:
# Seed the DB with a minimal filmography dataset.
# Safe to re-run — MERGE is idempotent.
driver.execute_query(
    "MERGE (m1:Movie {title: 'The Matrix', released: 1999})"
    " MERGE (m2:Movie {title: 'Speed', released: 1994})"
    " MERGE (p1:Person {name: 'Keanu Reeves', born: 1964})"
    " MERGE (p2:Person {name: 'Sandra Bullock', born: 1964})"
    " MERGE (p1)-[:ACTED_IN {role: 'Neo'}]->(m1)"
    " MERGE (p2)-[:ACTED_IN {role: 'Annie'}]->(m2)"
)
print("Seed data written (idempotent MERGE)")

Seed data written (idempotent MERGE)


In [11]:
# --- The ONLY change from the fake-session path above is this factory line. ---
# Everything else — query definition, adapter, call site — is identical.
live_executor = CypherExecutor(lambda: driver.session())

# Read: YAML-loaded query via adapter against the live DB
live_read_query = next(
    q
    for q in load_catalogue(YAML_CATALOGUE).queries()
    if q.query_id == "find_movie_by_title"
)
live_rows = live_executor.read(
    live_read_query, live_read_query.params_schema(title="The Matrix")
)

print(f"Live rows for 'The Matrix': {len(live_rows)}")
for row in live_rows:
    print(f"  {row}")

assert len(live_rows) >= 1
assert live_rows[0]["m.title"] == "The Matrix"

Live rows for 'The Matrix': 1
  {'m.title': 'The Matrix', 'm.released': 1999}


In [12]:
# Read: traversal query — actors for a movie
class ActorFilmParams(BaseModel):
    title: str


actor_query = CypherQuery(
    query_id="actors_for_movie",
    cypher_template=(
        "MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: $title}) RETURN p.name, p.born"
    ),
    params_schema=ActorFilmParams,
    identifiers_schema=NoIdentifiers,
)
actors = live_executor.read(actor_query, ActorFilmParams(title="The Matrix"))
print("Actors in 'The Matrix':")
for a in actors:
    print(f"  {a['p.name']} (born {a['p.born']})")

Actors in 'The Matrix':
  Keanu Reeves (born 1964)


In [13]:
# Write: CREATE a new movie node
create_query = CypherQuery(
    query_id="create_festival_movie",
    cypher_template="MERGE (m:Movie {title: $title, released: $released})",
    params_schema=CreateMovieParams,
    identifiers_schema=NoIdentifiers,
)
write_summary = live_executor.write(
    create_query, CreateMovieParams(title="Arrival", released=2016)
)
print(
    f"Write summary: nodes_created={write_summary.nodes_created}, properties_set={write_summary.properties_set}"
)

Write summary: nodes_created=1, properties_set=2


In [14]:
# Teardown: clean up nodes created by this notebook.
driver.execute_query("MATCH (m:Movie {title: 'Arrival'}) DETACH DELETE m")
driver.close()
print("Teardown complete. Driver closed.")

Teardown complete. Driver closed.


---

## Summary

| What was shown | Key point |
|---|---|
| YAML load â†’ `QueryCatalogue` | `load_query_catalogue` + `register_cypher_query` |
| `validate_query_catalogue` — clean | Same domain codes as typed path |
| Renamed-label ERROR | `QUERY_UNKNOWN_NODE_LABEL` — MP Phase-1 scenario closed |
| Stale `$param` ERROR | `QUERY_PARAM_ALIGNMENT_ERROR` — caught statically, not at runtime |
| Read round-trip (fake) | `CypherExecutor` directly â†’ `list[dict]` |
| Write round-trip (fake) | `CypherExecutor` directly â†’ `CypherWriteResultSummary` |
| Read round-trip (live) | Same code, `driver.session` factory instead of `FakeSession` |
| Write round-trip (live) | Same code, `CypherExecutor`, clean teardown |

**Switching fake â†’ live requires changing one line** — the factory passed to `CypherExecutor`.
Everything above the executor is untouched: same query definition, same adapter, same call site.

- Source: `src/orthograph/cypher/query.py`, `cypher/query_execution.py`
- E2E tests: `tests/cypher/test_query_e2e.py` (`pytest --neo4j`)
- ADR: `.agentic/decisions/027-simple-cypher-query-shared-validation.md`